# Face Occlusion Prediction - Data Challenge Telecom
**Task:** Predict the percentage of face occlusion from 224x224 cropped face images.  
**Provider:** IDEMIA Public Security  
**Metric:** Gender-balanced weighted MSE with fairness penalty  

---
## Strategy
- **Model:** EfficientNetV2-S (pretrained on ImageNet-21k) with regression head
- **Loss:** Custom weighted MSE matching the evaluation metric
- **Augmentations:** HorizontalFlip, Affine, ColorJitter, RandomErasing
- **Training:** OneCycleLR, mixed precision (AMP), 25 epochs, early stopping
- **Inference:** Test-time augmentation (TTA) with 4 transforms
- **Fairness:** Gender-aware validation monitoring + stratified splits

## 0. Setup & Configuration

In [ ]:
# Install dependencies (needed on Kaggle / Colab)
import subprocess, sys

def install_if_missing(package, import_name=None):
    try:
        __import__(import_name or package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

install_if_missing('timm')
install_if_missing('scikit-learn', 'sklearn')

In [ ]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# AMP compatibility: torch>=2.0 uses torch.amp, older uses torch.cuda.amp
try:
    from torch.amp import GradScaler, autocast
    AMP_AUTOCAST = lambda: autocast('cuda')
except ImportError:
    from torch.cuda.amp import GradScaler, autocast
    AMP_AUTOCAST = lambda: autocast()

import torchvision.transforms as T
import timm

from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore', category=FutureWarning)

# ============ CONFIG ============
IS_KAGGLE = os.path.exists('/kaggle/input')

CFG = {
    'seed': 42,
    'image_dir': '/kaggle/input/datasets/dragonballsuperhe/face-occlusion/face_occlusion_images/Crop_224_5fp_100K' if IS_KAGGLE else 'Crop_224_5fp_100K',
    'train_csv': '/kaggle/input/datasets/dragonballsuperhe/face-occlusion/occlusion_datasets/train.csv' if IS_KAGGLE else 'occlusion_datasets/train.csv',
    'test_csv': '/kaggle/input/datasets/dragonballsuperhe/face-occlusion/occlusion_datasets/test_students.csv' if IS_KAGGLE else 'occlusion_datasets/test_students.csv',
    'output_dir': '/kaggle/working' if IS_KAGGLE else '.',
    'img_size': 224,
    'model_name': 'tf_efficientnetv2_s.in21k_ft_in1k',
    'batch_size': 48,
    'epochs': 25,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'num_workers': 2 if IS_KAGGLE else 4,
    'n_folds': 5,
    'train_folds': [0, 1, 2, 3, 4],  # Full 5-fold CV for best score
    'use_tta': True,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Device: {CFG['device']}")
print(f"PyTorch: {torch.__version__} | timm: {timm.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['seed'])

## 1. Exploratory Data Analysis

In [ ]:
df_train = pd.read_csv(CFG['train_csv'])
df_test = pd.read_csv(CFG['test_csv'])

print(f"Train samples: {len(df_train):,}")
print(f"Test samples:  {len(df_test):,}")
print(f"\nTrain columns: {df_train.columns.tolist()}")
print(f"Test columns:  {df_test.columns.tolist()}")
print(f"\nNaN in train: {df_train.isna().sum().to_dict()}")
df_train.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall distribution
axes[0].hist(df_train['FaceOcclusion'], bins=100, alpha=0.7, color='steelblue')
axes[0].set_title(f'Train Occlusion Distribution (n={len(df_train):,})')
axes[0].set_xlabel('Face Occlusion')
axes[0].set_ylabel('Count')
axes[0].axvline(df_train['FaceOcclusion'].mean(), color='red', linestyle='--', label=f"mean={df_train['FaceOcclusion'].mean():.3f}")
axes[0].legend()

# By gender
for g, label, color in [(0.0, 'Female', 'coral'), (1.0, 'Male', 'steelblue')]:
    subset = df_train[df_train['gender'] == g]['FaceOcclusion']
    axes[1].hist(subset, bins=100, alpha=0.5, label=f'{label} (n={len(subset):,}, mean={subset.mean():.3f})', color=color)
axes[1].set_title('Occlusion by Gender')
axes[1].set_xlabel('Face Occlusion')
axes[1].legend()

# Gender counts
gender_counts = df_train['gender'].value_counts()
axes[2].bar(['Male (1.0)', 'Female (0.0)'], [gender_counts[1.0], gender_counts[0.0]], color=['steelblue', 'coral'])
axes[2].set_title('Gender Distribution')
axes[2].set_ylabel('Count')
for i, v in enumerate([gender_counts[1.0], gender_counts[0.0]]):
    axes[2].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nOcclusion statistics by gender:")
print(df_train.groupby('gender')['FaceOcclusion'].describe().round(4))

In [ ]:
# Visualize sample images at different occlusion levels
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
occlusion_bins = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8]

for i, occ_threshold in enumerate(occlusion_bins):
    row, col = i // 5, i % 5
    candidates = df_train[(df_train['FaceOcclusion'] >= occ_threshold) & 
                          (df_train['FaceOcclusion'] < occ_threshold + 0.05)]
    if len(candidates) > 0:
        sample = candidates.sample(1).iloc[0]
        img = Image.open(f"{CFG['image_dir']}/{sample['filename']}")
        axes[row, col].imshow(img)
        gender_str = 'M' if sample['gender'] == 1.0 else 'F'
        axes[row, col].set_title(f"Occ={sample['FaceOcclusion']:.2f} ({gender_str})")
    axes[row, col].axis('off')

plt.suptitle('Sample Images at Different Occlusion Levels', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Data Pipeline

In [ ]:
# Create stratified folds based on binned occlusion + gender
df_train = df_train.dropna().reset_index(drop=True)
df_test = df_test.dropna().reset_index(drop=True)

# Bin occlusion into categories for stratification
df_train['occ_bin'] = pd.cut(df_train['FaceOcclusion'], bins=10, labels=False)
df_train['stratify_col'] = df_train['gender'].astype(int).astype(str) + '_' + df_train['occ_bin'].astype(str)

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
df_train['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train['stratify_col'])):
    df_train.loc[val_idx, 'fold'] = fold

print("Fold distribution:")
print(df_train['fold'].value_counts().sort_index())
print(f"\nFold 0 gender split:")
print(df_train[df_train['fold'] == 0]['gender'].value_counts())

In [ ]:
# Transforms
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_train_transforms():
    return T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        T.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.9, 1.1)),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        T.RandomGrayscale(p=0.05),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        T.RandomErasing(p=0.3, scale=(0.02, 0.15)),
    ])

def get_val_transforms():
    return T.Compose([
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def get_tta_transforms():
    """Multiple TTA transforms for inference"""
    return [
        # Original
        T.Compose([T.ToTensor(), T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
        # Horizontal flip
        T.Compose([T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
        # Slight brightness up
        T.Compose([T.ColorJitter(brightness=(1.1, 1.1)), T.ToTensor(), T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
        # Slight brightness down
        T.Compose([T.ColorJitter(brightness=(0.9, 0.9)), T.ToTensor(), T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
    ]

In [ ]:
class FaceOcclusionDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{self.image_dir}/{row['filename']}").convert('RGB')

        if self.transform:
            img = self.transform(img)

        if self.is_test:
            return img, row['filename']

        label = torch.tensor(row['FaceOcclusion'], dtype=torch.float32)
        gender = torch.tensor(row['gender'], dtype=torch.float32)
        return img, label, gender

## 3. Model

In [ ]:
class OcclusionModel(nn.Module):
    def __init__(self, model_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        features = self.backbone(x)
        out = self.head(features)
        out = torch.sigmoid(out)  # Constrain to [0, 1]
        return out.squeeze(-1)

# Test model creation
model = OcclusionModel(CFG['model_name'])
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {CFG['model_name']}")
print(f"Total params: {n_params:,}")
print(f"Trainable params: {n_trainable:,}")
del model

## 4. Loss & Metric Functions

In [ ]:
class WeightedMSELoss(nn.Module):
    """Custom loss matching the evaluation metric: w_i = 1/30 + GT_i"""
    def __init__(self):
        super().__init__()

    def forward(self, pred, target):
        weights = 1.0 / 30.0 + target
        loss = weights * (pred - target) ** 2
        return loss.sum() / weights.sum()


def compute_error(pred, target):
    """Compute weighted MSE error for a subset"""
    weights = 1.0 / 30.0 + target
    return np.sum(weights * (pred - target) ** 2) / np.sum(weights)


def compute_score(pred, target, gender):
    """Compute the official evaluation score"""
    mask_f = gender == 0.0
    mask_m = gender == 1.0

    err_f = compute_error(pred[mask_f], target[mask_f]) if mask_f.sum() > 0 else 0.0
    err_m = compute_error(pred[mask_m], target[mask_m]) if mask_m.sum() > 0 else 0.0

    score = (err_f + err_m) / 2.0 + abs(err_f - err_m)
    return score, err_f, err_m

# Verify metric implementation matches example notebook
print("Metric functions defined.")

## 5. Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, device):
    model.train()
    criterion = WeightedMSELoss()
    running_loss = 0.0
    n_batches = 0

    pbar = tqdm(loader, desc='Training')
    for imgs, labels, genders in pbar:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        with AMP_AUTOCAST():
            preds = model(imgs)
            loss = criterion(preds, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({'loss': f'{running_loss / n_batches:.5f}'})

    return running_loss / n_batches


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    all_preds, all_targets, all_genders = [], [], []

    for imgs, labels, genders in tqdm(loader, desc='Validating'):
        imgs = imgs.to(device)
        with AMP_AUTOCAST():
            preds = model(imgs)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(labels.numpy())
        all_genders.append(genders.numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    all_genders = np.concatenate(all_genders)

    score, err_f, err_m = compute_score(all_preds, all_targets, all_genders)
    return score, err_f, err_m, all_preds

In [ ]:
def train_fold(fold, df_train, CFG):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    save_path = os.path.join(CFG['output_dir'], f'best_model_fold{fold}.pth')

    # Split data
    train_df = df_train[df_train['fold'] != fold]
    val_df = df_train[df_train['fold'] == fold]
    print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")
    print(f"Val gender: F={len(val_df[val_df['gender']==0.0]):,}, M={len(val_df[val_df['gender']==1.0]):,}")

    # Datasets & Loaders
    train_ds = FaceOcclusionDataset(train_df, CFG['image_dir'], get_train_transforms())
    val_ds = FaceOcclusionDataset(val_df, CFG['image_dir'], get_val_transforms())

    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                              num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'] * 2, shuffle=False,
                            num_workers=CFG['num_workers'], pin_memory=True)

    # Model
    model = OcclusionModel(CFG['model_name']).to(CFG['device'])

    # Optimizer & Scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    total_steps = len(train_loader) * CFG['epochs']
    warmup_steps = len(train_loader) * 2  # 2 epochs warmup
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=CFG['lr'] * 10,
        total_steps=total_steps,
        pct_start=warmup_steps / total_steps,
        anneal_strategy='cos',
        div_factor=10,
        final_div_factor=100,
    )
    scaler = GradScaler()

    # Training
    best_score = float('inf')
    best_preds = None
    patience = 5
    patience_counter = 0
    history = {'train_loss': [], 'val_score': [], 'err_f': [], 'err_m': []}

    for epoch in range(CFG['epochs']):
        print(f"\nEpoch {epoch+1}/{CFG['epochs']} | LR: {optimizer.param_groups[0]['lr']:.2e}")

        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, CFG['device'])
        val_score, err_f, err_m, val_preds = validate(model, val_loader, CFG['device'])

        history['train_loss'].append(train_loss)
        history['val_score'].append(val_score)
        history['err_f'].append(err_f)
        history['err_m'].append(err_m)

        print(f"Train Loss: {train_loss:.6f} | Val Score: {val_score:.6f} | Err_F: {err_f:.6f} | Err_M: {err_m:.6f}")

        if val_score < best_score:
            best_score = val_score
            best_preds = val_preds
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f"  >> New best score: {best_score:.6f} - Model saved!")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  >> Early stopping at epoch {epoch+1}")
                break

    # Cleanup
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\nBest validation score for fold {fold}: {best_score:.6f}")
    return best_score, history, val_df.index.tolist()

In [ ]:
# Train
all_scores = []
all_histories = []

for fold in CFG['train_folds']:
    score, history, val_idx = train_fold(fold, df_train, CFG)
    all_scores.append(score)
    all_histories.append(history)

print(f"\n{'='*60}")
print(f"RESULTS SUMMARY")
print(f"{'='*60}")
for i, (fold, score) in enumerate(zip(CFG['train_folds'], all_scores)):
    print(f"Fold {fold}: Score = {score:.6f}")
print(f"Mean Score: {np.mean(all_scores):.6f}")

In [ ]:
# Plot training history
for i, (fold, history) in enumerate(zip(CFG['train_folds'], all_histories)):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.set_title(f'Fold {fold} - Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.legend()

    ax2.plot(history['val_score'], label='Val Score', color='green')
    ax2.plot(history['err_f'], label='Err_F', color='coral', linestyle='--')
    ax2.plot(history['err_m'], label='Err_M', color='steelblue', linestyle='--')
    ax2.set_title(f'Fold {fold} - Validation Metrics')
    ax2.set_xlabel('Epoch')
    ax2.legend()

    plt.tight_layout()
    plt.show()

## 6. Inference with TTA

In [ ]:
@torch.no_grad()
def predict_with_tta(model, df, image_dir, device, tta_transforms=None):
    """Generate predictions with test-time augmentation"""
    model.eval()

    if tta_transforms is None:
        tta_transforms = [get_val_transforms()]

    all_preds = []
    for t_idx, transform in enumerate(tta_transforms):
        ds = FaceOcclusionDataset(df, image_dir, transform, is_test=True)
        loader = DataLoader(ds, batch_size=CFG['batch_size'] * 2, shuffle=False,
                            num_workers=CFG['num_workers'], pin_memory=True)

        preds = []
        for imgs, _ in tqdm(loader, desc=f'TTA {t_idx+1}/{len(tta_transforms)}'):
            imgs = imgs.to(device)
            with AMP_AUTOCAST():
                pred = model(imgs)
            preds.append(pred.cpu().numpy())
        all_preds.append(np.concatenate(preds))

    final_preds = np.mean(all_preds, axis=0)
    return final_preds

In [ ]:
# Load best model(s) and predict on test set
test_predictions = []

for fold in CFG['train_folds']:
    model = OcclusionModel(CFG['model_name'], pretrained=False).to(CFG['device'])
    ckpt_path = os.path.join(CFG['output_dir'], f'best_model_fold{fold}.pth')
    model.load_state_dict(torch.load(ckpt_path, map_location=CFG['device'], weights_only=True))

    if CFG['use_tta']:
        preds = predict_with_tta(model, df_test, CFG['image_dir'], CFG['device'], get_tta_transforms())
    else:
        preds = predict_with_tta(model, df_test, CFG['image_dir'], CFG['device'])

    test_predictions.append(preds)
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Average predictions across folds
final_predictions = np.mean(test_predictions, axis=0)

# Clip to valid range
final_predictions = np.clip(final_predictions, 0.0, 1.0)

print(f"Predictions shape: {final_predictions.shape}")
print(f"Predictions range: [{final_predictions.min():.4f}, {final_predictions.max():.4f}]")
print(f"Predictions mean:  {final_predictions.mean():.4f}")

In [ ]:
# Visualize prediction distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(final_predictions, bins=100, alpha=0.7, color='steelblue', label='Test predictions')
ax1.hist(df_train['FaceOcclusion'], bins=100, alpha=0.3, color='coral', label='Train GT', density=True)
ax1.set_title('Test Predictions Distribution')
ax1.set_xlabel('Face Occlusion')
ax1.legend()

ax2.hist(final_predictions, bins=100, alpha=0.7, color='steelblue')
ax2.set_title('Test Predictions (zoomed)')
ax2.set_xlabel('Face Occlusion')

plt.tight_layout()
plt.show()

## 7. Export Submission

In [ ]:
submission = pd.DataFrame({
    'filename': df_test['filename'],
    'FaceOcclusion': final_predictions,
    'gender': 'x'
})

out_path = os.path.join(CFG['output_dir'], 'test_predictions.csv')
submission.to_csv(out_path, index=False)
print(f"Submission saved: {out_path}")
print(f"Shape: {submission.shape}")
submission.head(10)

## 8. Advanced: Multi-Model Ensemble (Optional)

Uncomment and run the cell below to train a second model architecture and ensemble predictions for a stronger submission.

In [ ]:
# ============ ENSEMBLE: Train a second model ============
CFG2 = CFG.copy()
CFG2['model_name'] = 'convnextv2_tiny.fcmae_ft_in22k_in1k'
CFG2['lr'] = 5e-5
CFG2['batch_size'] = 24  # Smaller batch for larger model

all_scores_2 = []
for fold in CFG2['train_folds']:
    score, history, _ = train_fold(fold, df_train, CFG2)
    all_scores_2.append(score)
    src = os.path.join(CFG2['output_dir'], f'best_model_fold{fold}.pth')
    dst = os.path.join(CFG2['output_dir'], f'best_model_convnext_fold{fold}.pth')
    os.rename(src, dst)

print(f"ConvNeXt Mean Score: {np.mean(all_scores_2):.6f}")

# Generate ConvNeXt predictions
test_preds_2 = []
for fold in CFG2['train_folds']:
    model = OcclusionModel(CFG2['model_name'], pretrained=False).to(CFG2['device'])
    ckpt = os.path.join(CFG2['output_dir'], f'best_model_convnext_fold{fold}.pth')
    model.load_state_dict(torch.load(ckpt, map_location=CFG2['device'], weights_only=True))
    preds = predict_with_tta(model, df_test, CFG2['image_dir'], CFG2['device'], get_tta_transforms())
    test_preds_2.append(preds)
    del model; gc.collect(); torch.cuda.empty_cache()

preds_convnext = np.mean(test_preds_2, axis=0)

# Ensemble: weighted average
ensemble_preds = 0.5 * final_predictions + 0.5 * preds_convnext
ensemble_preds = np.clip(ensemble_preds, 0.0, 1.0)

submission_ens = pd.DataFrame({
    'filename': df_test['filename'],
    'FaceOcclusion': ensemble_preds,
    'gender': 'x'
})
out_path = os.path.join(CFG2['output_dir'], 'test_predictions.csv')
submission_ens.to_csv(out_path, index=False)
print(f"Ensemble submission saved: {out_path}")
print(f"Shape: {submission_ens.shape}")
submission_ens.head(10)

In [ ]:
# ============ FINAL SUMMARY ============
print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"\nEfficientNetV2-S ({CFG['model_name']}):")
for fold, score in zip(CFG['train_folds'], all_scores):
    print(f"  Fold {fold}: {score:.6f}")
print(f"  Mean CV: {np.mean(all_scores):.6f}")

print(f"\nConvNeXtV2-Tiny ({CFG2['model_name']}):")
for fold, score in zip(CFG2['train_folds'], all_scores_2):
    print(f"  Fold {fold}: {score:.6f}")
print(f"  Mean CV: {np.mean(all_scores_2):.6f}")

print(f"\nEnsemble (50/50 average):")
print(f"  Test predictions range: [{ensemble_preds.min():.4f}, {ensemble_preds.max():.4f}]")
print(f"  Test predictions mean:  {ensemble_preds.mean():.4f}")
print(f"\nSubmission: {out_path} ({len(submission_ens)} rows)")
print("=" * 60)

---
## Summary

| Component | Choice | Rationale |
|---|---|---|
| Backbone | EfficientNetV2-S (ImageNet-21k pretrained) | Strong features, fits in 16GB VRAM |
| Head | Dropout + FC(256) + ReLU + Dropout + FC(1) + Sigmoid | Regularized, output in [0,1] |
| Loss | Weighted MSE (w_i = 1/30 + GT_i) | Matches evaluation metric exactly |
| Optimizer | AdamW (lr=1e-4, wd=1e-4) | Standard for fine-tuning |
| Scheduler | OneCycleLR (2-epoch warmup, cosine anneal) | Aggressive but effective |
| Augmentation | HFlip, Affine, ColorJitter, Grayscale, RandomErasing | Diverse without destroying face structure |
| Validation | Stratified 5-fold (by gender + occlusion bin) | Fair evaluation |
| Inference | TTA (4 transforms: original, flip, brightness+/-) | Free accuracy boost |
| Training | Mixed precision (AMP) + gradient clipping | Memory efficient, stable training |